In [0]:
# MAGIC %python
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

catalog_name = "emissor_a"
spark.sql(f"USE CATALOG {catalog_name}")

# -------------------------------------------------------------------------
# 1. CAMADA SILVER: Calculando métricas de comportamento por fatura/cliente
# -------------------------------------------------------------------------
# Simulando dados limpos de faturas mensais de cartões de crédito
silver_data = [
    ("C001", "FAT-2026-07", "2026-07-10", "2026-07-08", 1200.00, 1200.00, 0.45, 12),
    ("C001", "FAT-2026-08", "2026-08-10", "2026-08-12", 1500.00, 1500.00, 0.60, 15), # 2 dias de atraso
    ("C001", "FAT-2026-09", "2026-09-10", "2026-09-10", 2200.00, 500.00,  0.85, 20), # Pagou parcial (mínimo)
    ("C002", "FAT-2026-08", "2026-08-15", "2026-08-15", 300.00,  300.00,  0.15, 4),
    ("C002", "FAT-2026-09", "2026-09-15", "2026-09-14", 450.00,  450.00,  0.20, 6)
]

df_silver = spark.createDataFrame(silver_data, [
    "cliente_id", "fatura_id", "data_vencimento", "data_pagamento", 
    "valor_fatura", "valor_pago", "utilizacao_limite", "qtd_transacoes_mes"
])

# Enriquecendo com regras de comportamento na Silver
df_silver_enriched = df_silver.withColumn(
    "dias_atraso", 
    F.greatest(F.datediff(F.col("data_pagamento"), F.col("data_vencimento")), F.lit(0))
).withColumn(
    "pagou_integral", 
    F.col("valor_pago") >= F.col("valor_fatura")
)

df_silver_enriched.write.format("delta").mode("overwrite").saveAsTable("silver.faturas_comportamento")
print("Camada Silver atualizada com colunas de comportamento.")

# -------------------------------------------------------------------------
# 2. CAMADA GOLD: Agregando o Behavior Score (Features consolidadas por cliente)
# -------------------------------------------------------------------------
df_silver_read = spark.table("silver.faturas_comportamento")

df_gold_behavior_score = df_silver_read.groupBy("cliente_id").agg(
    F.max("dias_atraso").alias("max_dias_atraso_historico"),
    F.sum(F.when(F.col("dias_atraso") > 0, 1).otherwise(0)).alias("qtd_faturas_atrasadas"),
    F.avg("utilizacao_limite").alias("media_utilizacao_limite"),
    F.sum(F.when(F.col("pagou_integral") == False, 1).otherwise(0)).alias("qtd_vezes_pagamento_parcial"),
    F.sum("qtd_transacoes_mes").alias("total_transacoes_periodo"),
    F.max("data_vencimento").alias("ultima_fatura_analisada")
).withColumn(
    # Exemplo simples de regra de risco base para demonstração
    "faixa_risco_preliminar",
    F.when(F.col("max_dias_atraso_historico") > 5, "Alto Risco")
     .when(F.col("qtd_vezes_pagamento_parcial") > 0, "Médio Risco")
     .otherwise("Baixo Risco")
)

df_gold_behavior_score.write.format("delta").mode("overwrite").saveAsTable("gold.behavior_score_clientes")
print("Camada Gold (Behavior Score) gerada com sucesso!")